# Обучение модели предсказания кредитного дефолта

Этот ноутбук содержит процесс подготовки данных, обучения и валидации GRU-модели для предсказания риска дефолта клиентов.

In [1]:
import sys
from pathlib import Path

__ROOT__ = Path.cwd()
if not (__ROOT__ / "data").exists() and (__ROOT__.parent / "data").exists():
    __ROOT__ = __ROOT__.parent

DATABASE_PATH = __ROOT__ / "data/database.db"
SCALER_PATH = __ROOT__ / "scaler.json"

ID_COL = "client_id"
TARGET_COL = "default"
SCALER_COLS = ["pay_amt", "bill_amt", "limit_bal"]

if str(__ROOT__) not in sys.path:
    sys.path.append(str(__ROOT__))

## 1. Загрузка библиотек и модулей проекта

In [2]:
import sqlite3
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Импорт собственных модулей проекта
from model import CreditDefaultPredictor, preprocess, normalize, prepare_dataset
from model.normalization import fit_and_save_scaler

## 2. Загрузка и предобработка данных (ETL)

Загружаем данные из SQLite базы данных, объединяем информацию о клиентах и истории платежей, производим базовую чистку признаков и делим выборку на обучающую (train) и тестовую (test) с сохранением баланса классов (стратификация).

In [3]:
with sqlite3.connect(DATABASE_PATH) as conn:
    client_df = pd.read_sql_query("SELECT * FROM clients", conn)
    history_df = pd.read_sql_query("SELECT * FROM payment_history", conn)

df = pd.merge(client_df, history_df, on="client_id")
df = preprocess(df)

client_data = df[[ID_COL, TARGET_COL]].drop_duplicates()
train_ids, test_ids = train_test_split(
    client_data[ID_COL],
    test_size=0.2,
    stratify=client_data[TARGET_COL],
    random_state=42
)

train_df = df[df[ID_COL].isin(train_ids)]
test_df = df[df[ID_COL].isin(test_ids)]

fit_and_save_scaler(train_df, SCALER_COLS, SCALER_PATH)
train_df = normalize(train_df, SCALER_COLS, SCALER_PATH)
test_df = normalize(test_df, SCALER_COLS, SCALER_PATH)

## 3. Подготовка датасетов для PyTorch

Используем общую функцию `prepare_dataset` из модуля `model.dataset` для преобразования временных последовательностей клиентов в 3D тензоры (формат `[batch_size, seq_len, num_features]`) и создания `CreditDataset`.

In [4]:
train_dataset = prepare_dataset(train_df, id_col=ID_COL, target_col=TARGET_COL)
test_dataset = prepare_dataset(test_df, id_col=ID_COL, target_col=TARGET_COL)

## 4. Обучение модели

Инициализируем GRU-модель, лосс-функцию (с учетом дисбаланса классов) и оптимизатор Adam.  
В качестве лосс-функции используем `BCEWithLogitsLoss` с параметром `pos_weight`, который задается как отношение негативных примеров к позитивным.

In [5]:
BATCH_SIZE = 10
LEARNING_RATE = 0.001

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = CreditDefaultPredictor()

pos_weight = torch.tensor([78.0 / 22.0])
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

print("Starting training...")
for epoch in range(32):
    model.train()
    epoch_losses = []
    for features, labels in train_loader:
        features = features.float()
        labels = labels.float()
        
        optimizer.zero_grad()
        outputs = model(features)
        
        loss_val = loss_fn(outputs, labels)
        
        epoch_losses.append(loss_val.item())
        
        loss_val.backward()
        optimizer.step()
        
    avg_loss = sum(epoch_losses) / len(epoch_losses)
    print(f"Epoch {epoch+1:02d}/32 - Average Loss: {avg_loss:.4f}")

Starting training...
Epoch 01/32 - Average Loss: 0.9230
Epoch 02/32 - Average Loss: 0.8989
Epoch 03/32 - Average Loss: 0.8920
Epoch 04/32 - Average Loss: 0.8882
Epoch 05/32 - Average Loss: 0.8864
Epoch 06/32 - Average Loss: 0.8836
Epoch 07/32 - Average Loss: 0.8829
Epoch 08/32 - Average Loss: 0.8818
Epoch 09/32 - Average Loss: 0.8799
Epoch 10/32 - Average Loss: 0.8776
Epoch 11/32 - Average Loss: 0.8758
Epoch 12/32 - Average Loss: 0.8759
Epoch 13/32 - Average Loss: 0.8747
Epoch 14/32 - Average Loss: 0.8727
Epoch 15/32 - Average Loss: 0.8723
Epoch 16/32 - Average Loss: 0.8701
Epoch 17/32 - Average Loss: 0.8689
Epoch 18/32 - Average Loss: 0.8665
Epoch 19/32 - Average Loss: 0.8655
Epoch 20/32 - Average Loss: 0.8635
Epoch 21/32 - Average Loss: 0.8625
Epoch 22/32 - Average Loss: 0.8594
Epoch 23/32 - Average Loss: 0.8576
Epoch 24/32 - Average Loss: 0.8557
Epoch 25/32 - Average Loss: 0.8518
Epoch 26/32 - Average Loss: 0.8495
Epoch 27/32 - Average Loss: 0.8465
Epoch 28/32 - Average Loss: 0.8428

## 5. Валидация модели на тестовом наборе данных

Оцениваем качество обученной модели на отложенной тестовой выборке с расчетом метрики ROC AUC и классификационного отчета (Classification Report).

In [6]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for features, labels in test_loader:
        features = features.float()
        outputs = model(features)
        
        probs = torch.sigmoid(outputs)
        all_preds.extend(probs.numpy())
        all_targets.extend(labels.numpy())

roc_auc = roc_auc_score(all_targets, all_preds)
print(f"Test ROC AUC: {roc_auc:.4f}\n")

# Бинаризация предсказаний с порогом 0.5 для отчета классификации
binary_preds = [1 if p >= 0.5 else 0 for p in all_preds]
print("Classification Report:")
print(classification_report(all_targets, binary_preds, target_names=["Non-default", "Default"]))

Test ROC AUC: 0.7514

Classification Report:
              precision    recall  f1-score   support

 Non-default       0.87      0.80      0.84      4673
     Default       0.46      0.58      0.51      1327

    accuracy                           0.75      6000
   macro avg       0.66      0.69      0.67      6000
weighted avg       0.78      0.75      0.76      6000

